In [0]:
-- 1
SELECT 
  `date`,
  SUM(`paid_price`) as daily_revenue
FROM `itransition`.`default`.`parquet_df_with_price`
WHERE `date` IS NOT NULL AND `paid_price` IS NOT NULL
GROUP BY `date`
ORDER BY daily_revenue DESC
LIMIT 5;
-- 2
WITH duplicates AS (
    SELECT STRING_AGG(DISTINCT CAST(id AS STRING), ',') AS user_ids
    FROM `itransition`.`default`.`csv_df` GROUP BY phone, email, address
    HAVING COUNT(DISTINCT name) > 1
    UNION ALL
    SELECT STRING_AGG(DISTINCT CAST(id AS STRING), ',') AS user_ids
    FROM `itransition`.`default`.`csv_df` GROUP BY name, email, address
    HAVING COUNT(DISTINCT phone) > 1
    UNION ALL
    SELECT STRING_AGG(DISTINCT CAST(id AS STRING), ',') AS user_ids
    FROM `itransition`.`default`.`csv_df` GROUP BY name, phone, address
    HAVING COUNT(DISTINCT email) > 1
    UNION ALL
    SELECT STRING_AGG(DISTINCT CAST(id AS STRING), ',') AS user_ids
    FROM `itransition`.`default`.`csv_df` GROUP BY name, phone, email
    HAVING COUNT(DISTINCT address) > 1
),
groups AS (
    SELECT
        user_ids,
        SIZE(SPLIT(user_ids, ',')) AS group_size
    FROM duplicates
)
SELECT
    (SELECT COUNT(*) FROM `itransition`.`default`.`csv_df`)        AS total_records,     -- 5
    SUM(group_size)                       AS duplicate_records, -- 4
    COUNT(*)                              AS groups_count,      -- 2
    (SELECT COUNT(*) FROM `itransition`.`default`.`csv_df`)
        - SUM(group_size)
        + COUNT(*)                        AS unique_real_users  -- 5-4+2=3
FROM groups;

-- 3
SELECT 
  COUNT(DISTINCT `:author`) as unique_author_sets
FROM `itransition`.`default`.`yaml_df`
WHERE `:author` IS NOT NULL;


-- 4
SELECT 
  y.`:author`,
  SUM(p.`quantity`) as total_books_sold
FROM `itransition`.`default`.`parquet_df_with_price` p
JOIN `itransition`.`default`.`yaml_df` y ON p.`book_id` = y.`:id`
WHERE y.`:author` IS NOT NULL
GROUP BY y.`:author`
ORDER BY total_books_sold DESC
LIMIT 10;

-- 5
WITH user_pairs AS (
  SELECT 
    c1.`id` as user_id_1,
    c2.`id` as user_id_2,
    CASE WHEN c1.`name` = c2.`name` AND c1.`name` IS NOT NULL AND c1.`name` != '' AND c1.`name` != 'NULL' THEN 1 ELSE 0 END as name_match,
    CASE WHEN c1.`address` = c2.`address` AND c1.`address` IS NOT NULL AND c1.`address` != '' AND c1.`address` != 'NULL' AND c1.`address` != ' ' THEN 1 ELSE 0 END as address_match,
    CASE WHEN c1.`phone` = c2.`phone` AND c1.`phone` IS NOT NULL AND c1.`phone` != '' AND c1.`phone` != 'NULL' THEN 1 ELSE 0 END as phone_match,
    CASE WHEN c1.`email` = c2.`email` AND c1.`email` IS NOT NULL AND c1.`email` != '' AND c1.`email` != 'NULL' THEN 1 ELSE 0 END as email_match
  FROM `itransition`.`default`.`csv_df` c1
  CROSS JOIN `itransition`.`default`.`csv_df` c2
  WHERE c1.`id` < c2.`id`
),
valid_pairs AS (
  SELECT 
    user_id_1,
    user_id_2,
    (name_match + address_match + phone_match + email_match) as total_matches
  FROM user_pairs
  WHERE (name_match + address_match + phone_match + email_match) = 3  -- Bu qator muhim!
),
user_spending AS (
  SELECT `user_id`, SUM(`paid_price` * `quantity`) as total_spending
  FROM `itransition`.`default`.`parquet_df_with_price`
  GROUP BY `user_id`
)
SELECT 
  vp.user_id_1,
  vp.user_id_2,
  vp.total_matches,
  COALESCE(us1.total_spending, 0) as spending_1,
  COALESCE(us2.total_spending, 0) as spending_2,
  COALESCE(us1.total_spending, 0) + COALESCE(us2.total_spending, 0) as combined_spending
FROM valid_pairs vp
LEFT JOIN user_spending us1 ON vp.user_id_1 = us1.user_id
LEFT JOIN user_spending us2 ON vp.user_id_2 = us2.user_id
where total_matches=3
ORDER BY combined_spending DESC
LIMIT 10;